# ROLLUP, CUBE, and GROUPING SETS

## Overview
These are extensions to `GROUP BY` that compute **multiple levels of aggregation in a single query** — subtotals, grand totals, and cross-tabulation totals — without `UNION ALL`.

| Extension | What it produces |
|---|---|
| `GROUP BY ROLLUP(a, b)` | Groups by (a, b), then (a), then () — hierarchical subtotals |
| `GROUP BY CUBE(a, b)` | All possible combinations: (a,b), (a), (b), () |
| `GROUP BY GROUPING SETS((a,b),(a),())` | Exactly the combinations you specify |

**PostgreSQL:** All three are fully supported natively.

**SQLite:** None of these are supported. The standard workaround is `UNION ALL` of multiple `GROUP BY` queries at each level. This notebook demonstrates both the PostgreSQL syntax and the portable SQLite equivalent.

**The `GROUPING()` function** (PostgreSQL): returns 1 when the column was "rolled up" (i.e. the NULL represents a subtotal, not a genuine NULL), and 0 when it's a real group value. Essential for distinguishing subtotal NULLs from data NULLs.

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT);
CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, province TEXT, opened_date TEXT, status TEXT);
CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT, flagged INTEGER);
CREATE TABLE loans (loan_id INTEGER PRIMARY KEY, customer_id INTEGER, loan_type TEXT, principal REAL, rate_pct REAL, term_months INTEGER, issued_date TEXT, status TEXT);
INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),(2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),(4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),(6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),(8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),(10,'Marcus','Okafor','Retail','ON');
INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),(102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),(104,2,'Investment','BC','2021-01-10','Active'),
  (105,3,'Chequing','ON','2018-05-20','Active'),(106,3,'Investment','ON','2018-05-20','Active'),
  (107,3,'Savings','ON','2022-11-01','Active'),(108,4,'Chequing','NB','2015-09-30','Active'),
  (109,4,'Loan','NB','2020-04-01','Closed'),(110,5,'Chequing','BC','2021-06-15','Active'),
  (111,5,'Savings','BC','2021-06-15','Suspended'),(112,6,'Chequing','AB','2017-11-22','Active'),
  (113,7,'Investment','ON','2016-03-08','Active'),(114,7,'Chequing','ON','2016-03-08','Active'),
  (115,8,'Chequing','QC','2023-01-05','Active'),(116,9,'Chequing','BC','2022-08-19','Active'),
  (117,9,'Investment','BC','2022-08-19','Active'),(118,10,'Chequing','ON','2020-12-01','Active'),
  (119,10,'Savings','ON','2020-12-01','Active');
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),(1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),(1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),(1006,102,'2023-01-15','Deposit',500.00,'Transfer',0),
  (1007,102,'2023-03-10','Interest',12.50,'Interest',0),(1008,103,'2023-01-08','Deposit',15000.00,'Payroll',0),
  (1009,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),(1010,103,'2023-02-08','Deposit',15000.00,'Payroll',0),
  (1011,104,'2023-01-31','Deposit',5000.00,'Transfer',0),(1012,105,'2023-01-10','Deposit',22000.00,'Payroll',0),
  (1013,105,'2023-01-28','Withdrawal',-8500.00,'Rent',0),(1014,105,'2023-02-10','Deposit',22000.00,'Payroll',0),
  (1015,106,'2023-02-01','Deposit',50000.00,'Investment',0),(1016,108,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1017,108,'2023-01-19','Withdrawal',-700.00,'Rent',0),(1018,108,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1019,110,'2023-01-15','Deposit',8500.00,'Payroll',0),(1020,110,'2023-02-01','Withdrawal',-2100.00,'Rent',0),
  (1021,112,'2023-01-20','Deposit',3800.00,'Payroll',0),(1022,112,'2023-02-10','Fee',-25.00,'Fee',1),
  (1023,113,'2023-01-05','Deposit',75000.00,'Investment',0),(1024,114,'2023-01-12','Deposit',12000.00,'Payroll',0),
  (1025,114,'2023-02-12','Deposit',12000.00,'Payroll',0),(1026,115,'2023-01-10','Deposit',2800.00,'Payroll',0),
  (1027,115,'2023-01-28','Withdrawal',-650.00,'Groceries',0),(1028,116,'2023-01-18','Deposit',9200.00,'Payroll',0),
  (1029,117,'2023-02-05','Deposit',10000.00,'Investment',0),(1030,118,'2023-01-22','Deposit',4500.00,'Payroll',0),
  (1031,118,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),(1032,119,'2023-02-20','Interest',18.75,'Interest',0),
  (1033,101,'2023-03-05','Deposit',4200.00,'Payroll',NULL),(1034,103,'2023-03-08','Deposit',15000.00,'Payroll',0),
  (1035,112,'2023-03-15','Withdrawal',-450.00,'Groceries',1);
INSERT INTO loans VALUES
  (201,1,'Personal',15000.00,7.5,36,'2021-06-01','Current'),(202,2,'Mortgage',480000.00,4.2,300,'2020-07-15','Current'),
  (203,3,'HELOC',100000.00,6.1,60,'2019-02-28','Current'),(204,4,'Auto',28000.00,5.9,60,'2020-04-01','Paid Off'),
  (205,5,'Mortgage',390000.00,4.8,300,'2021-06-20','Current'),(206,6,'Personal',8000.00,9.2,24,'2022-03-10','Delinquent'),
  (207,7,'Mortgage',750000.00,3.9,300,'2018-09-01','Current'),(208,8,'Auto',22000.00,6.5,48,'2023-01-15','Current'),
  (209,9,'Personal',12000.00,8.1,36,'2022-08-25','Current'),(210,10,'Auto',35000.00,5.4,60,'2021-03-01','Current'),
  (211,3,'Personal',25000.00,6.8,48,'2020-11-15','Paid Off'),(212,7,'HELOC',200000.00,5.5,60,'2022-06-01','Current');
""")
conn.commit()
print('Finance schema ready.')


Finance schema ready.


---
## ROLLUP — hierarchical subtotals (SQLite workaround)

In [2]:
# ROLLUP equivalent: transactions by province → account_type → grand total
# PostgreSQL: GROUP BY ROLLUP(c.province, a.account_type)
# SQLite: UNION ALL at each rollup level

print('=== ROLLUP equivalent: province > account_type > grand total ===')
q = """
-- Level 1: province + account_type detail
SELECT
    c.province      AS province,
    a.account_type  AS account_type,
    COUNT(t.txn_id) AS txn_count,
    ROUND(SUM(t.amount), 2) AS total_amount,
    '  detail'      AS rollup_level
FROM transactions t
JOIN accounts a ON t.account_id = a.account_id
JOIN customers c ON a.customer_id = c.customer_id
GROUP BY c.province, a.account_type

UNION ALL

-- Level 2: province subtotal
SELECT
    c.province,
    '(all types)'   AS account_type,
    COUNT(t.txn_id),
    ROUND(SUM(t.amount), 2),
    ' subtotal'
FROM transactions t
JOIN accounts a ON t.account_id = a.account_id
JOIN customers c ON a.customer_id = c.customer_id
GROUP BY c.province

UNION ALL

-- Level 3: grand total
SELECT
    '(all provinces)',
    '(all types)',
    COUNT(txn_id),
    ROUND(SUM(amount), 2),
    'grand total'
FROM transactions

ORDER BY province, account_type, rollup_level
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== ROLLUP equivalent: province > account_type > grand total ===
       province account_type  txn_count  total_amount rollup_level
(all provinces)  (all types)         35     282456.25  grand total
             AB  (all types)          3       3325.00     subtotal
             AB     Chequing          3       3325.00       detail
             BC  (all types)          9      72400.00     subtotal
             BC     Chequing          7      57400.00       detail
             BC   Investment          2      15000.00       detail
             NB  (all types)         11      16662.50     subtotal
             NB     Chequing          9      16150.00       detail
             NB      Savings          2        512.50       detail
             ON  (all types)         10     187918.75     subtotal
             ON     Chequing          7      62900.00       detail
             ON   Investment          2     125000.00       detail
             ON      Savings          1         18.75       deta

---
## PostgreSQL ROLLUP and GROUPING() — reference syntax

In [3]:
print('=== PostgreSQL ROLLUP syntax (reference) ===')
print("""
SELECT
    c.province,
    a.account_type,
    COUNT(t.txn_id)         AS txn_count,
    ROUND(SUM(t.amount), 2) AS total_amount,
    GROUPING(c.province)    AS is_province_rollup,   -- 1 = this is a subtotal row
    GROUPING(a.account_type) AS is_type_rollup
FROM transactions AS t
JOIN accounts  AS a ON t.account_id  = a.account_id
JOIN customers AS c ON a.customer_id = c.customer_id
GROUP BY ROLLUP(c.province, a.account_type)
ORDER BY c.province NULLS LAST, a.account_type NULLS LAST;

-- GROUPING() lets you label the subtotal rows cleanly:
SELECT
    CASE GROUPING(c.province)
        WHEN 1 THEN '(All Provinces)' ELSE c.province END  AS province,
    CASE GROUPING(a.account_type)
        WHEN 1 THEN '(All Types)'     ELSE a.account_type  END AS account_type,
    COUNT(*) AS txns
FROM transactions AS t
JOIN accounts  AS a ON t.account_id  = a.account_id
JOIN customers AS c ON a.customer_id = c.customer_id
GROUP BY ROLLUP(c.province, a.account_type);
""")


=== PostgreSQL ROLLUP syntax (reference) ===

SELECT
    c.province,
    a.account_type,
    COUNT(t.txn_id)         AS txn_count,
    ROUND(SUM(t.amount), 2) AS total_amount,
    GROUPING(c.province)    AS is_province_rollup,   -- 1 = this is a subtotal row
    GROUPING(a.account_type) AS is_type_rollup
FROM transactions AS t
JOIN accounts  AS a ON t.account_id  = a.account_id
JOIN customers AS c ON a.customer_id = c.customer_id
GROUP BY ROLLUP(c.province, a.account_type)
ORDER BY c.province NULLS LAST, a.account_type NULLS LAST;

-- GROUPING() lets you label the subtotal rows cleanly:
SELECT
    CASE GROUPING(c.province)
        WHEN 1 THEN '(All Provinces)' ELSE c.province END  AS province,
    CASE GROUPING(a.account_type)
        WHEN 1 THEN '(All Types)'     ELSE a.account_type  END AS account_type,
    COUNT(*) AS txns
FROM transactions AS t
JOIN accounts  AS a ON t.account_id  = a.account_id
JOIN customers AS c ON a.customer_id = c.customer_id
GROUP BY ROLLUP(c.province, a.acco

---
## CUBE — all combinations (SQLite UNION ALL workaround)

In [4]:
# CUBE on (province, loan_type) — all four grouping combinations
print('=== CUBE equivalent: all (province x loan_type) combinations ===')
q = """
SELECT c.province, l.loan_type,
       COUNT(*) AS loans,
       ROUND(SUM(l.principal),2) AS total_principal,
       'province+type' AS group_level
FROM loans l JOIN customers c ON l.customer_id = c.customer_id
GROUP BY c.province, l.loan_type

UNION ALL
SELECT c.province, '(all types)',
       COUNT(*), ROUND(SUM(l.principal),2), 'province only'
FROM loans l JOIN customers c ON l.customer_id = c.customer_id
GROUP BY c.province

UNION ALL
SELECT '(all provinces)', l.loan_type,
       COUNT(*), ROUND(SUM(l.principal),2), 'type only'
FROM loans l JOIN customers c ON l.customer_id = c.customer_id
GROUP BY l.loan_type

UNION ALL
SELECT '(all provinces)', '(all types)',
       COUNT(*), ROUND(SUM(l.principal),2), 'grand total'
FROM loans l JOIN customers c ON l.customer_id = c.customer_id

ORDER BY group_level, province, loan_type
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== CUBE equivalent: all (province x loan_type) combinations ===
       province   loan_type  loans  total_principal   group_level
(all provinces) (all types)     12        2065000.0   grand total
             AB (all types)      1           8000.0 province only
             BC (all types)      3         882000.0 province only
             NB (all types)      2          43000.0 province only
             ON (all types)      5        1110000.0 province only
             QC (all types)      1          22000.0 province only
             AB    Personal      1           8000.0 province+type
             BC    Mortgage      2         870000.0 province+type
             BC    Personal      1          12000.0 province+type
             NB        Auto      1          28000.0 province+type
             NB    Personal      1          15000.0 province+type
             ON        Auto      1          35000.0 province+type
             ON       HELOC      2         300000.0 province+type
           

---
## GROUPING SETS — specifying exact combinations

In [5]:
# GROUPING SETS: only the groupings that are useful for a specific report
# Example: segment total + province total + grand total (skip segment×province detail)
print('=== GROUPING SETS equivalent: segment totals + province totals + grand total ===')
q = """
SELECT 'By Segment' AS grouping_type, c.segment AS label,
       COUNT(l.loan_id) AS loans, ROUND(SUM(l.principal),2) AS principal
FROM loans l JOIN customers c ON l.customer_id = c.customer_id
GROUP BY c.segment

UNION ALL
SELECT 'By Province', c.province,
       COUNT(l.loan_id), ROUND(SUM(l.principal),2)
FROM loans l JOIN customers c ON l.customer_id = c.customer_id
GROUP BY c.province

UNION ALL
SELECT 'Grand Total', '(all)',
       COUNT(l.loan_id), ROUND(SUM(l.principal),2)
FROM loans l JOIN customers c ON l.customer_id = c.customer_id

ORDER BY grouping_type, principal DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print('PostgreSQL GROUPING SETS syntax:')
print("""
GROUP BY GROUPING SETS (
    (c.segment),
    (c.province),
    ()               -- grand total
)
""")

conn.close()


=== GROUPING SETS equivalent: segment totals + province totals + grand total ===
grouping_type  label  loans  principal
  By Province     ON      5  1110000.0
  By Province     BC      3   882000.0
  By Province     NB      2    43000.0
  By Province     QC      1    22000.0
  By Province     AB      1     8000.0
   By Segment Wealth      4  1075000.0
   By Segment    SME      3   882000.0
   By Segment Retail      5   108000.0
  Grand Total  (all)     12  2065000.0

PostgreSQL GROUPING SETS syntax:

GROUP BY GROUPING SETS (
    (c.segment),
    (c.province),
    ()               -- grand total
)



---
## Common Pitfalls

**1. ROLLUP NULLs are indistinguishable from data NULLs without GROUPING()**
After a `GROUP BY ROLLUP(province, account_type)`, a NULL in `province` could mean "this is a grand total row" or "the province was genuinely NULL in the data." Use `GROUPING(province)` to tell them apart — it returns 1 for rollup-generated NULLs and 0 for real NULLs.

**2. ROLLUP order is hierarchical — changing it changes the subtotals**
`ROLLUP(province, account_type)` produces subtotals by province (collapsing account_type), then a grand total. `ROLLUP(account_type, province)` produces subtotals by account_type (collapsing province). The first column is the outermost grouping level.

**3. CUBE on high-cardinality columns explodes row count**
`CUBE(province, account_type, segment)` produces 2³ = 8 grouping combinations. With 5 columns it's 32. With large numbers of distinct values per column, CUBE results can be enormous. Use `GROUPING SETS` to specify only the combinations you actually need.

**4. SQLite UNION ALL workaround requires repeating the join logic**
Each level of the UNION ALL must independently join all needed tables. This is verbose and error-prone — a typo in one leg's WHERE clause produces subtotals that don't match the detail rows. Consider materialising the joined data into a CTE first, then running the UNION ALL against the CTE.

**5. Mixing ROLLUP subtotal rows into downstream aggregations**
If you store ROLLUP output in a staging table or use it as a subquery, the grand total row will double-count if you `SUM()` over all rows. Always filter on `GROUPING()` flags or the label column before aggregating further.

**6. HAVING on ROLLUP results filters both detail and subtotal rows**
`HAVING SUM(principal) > 100000` after a ROLLUP removes any grouping level (including the grand total) whose sum is under the threshold. If you only want to filter the detail rows, filter in a subquery before applying ROLLUP.


---
*sql_methods_library - Samantha McGarrigle*